## Part 3 - Model Training

Pipeline:
1. Load processed features CSV (output of notebook 2)
2. Per-ticker chronological split → scale → build LSTM sequences
3. Validate splits (no data leakage, correct shapes)
4. Train LSTM model
5. Evaluate on test set

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from src.data.split import train_val_test_split, scale_splits, make_sequences

### 1. Load Processed Features

In [ ]:
features_path = Path("/Users/cps/DL4AI-240166-project-1/notebooks/data/nasdaq/csv/tech_nasdaq_stock_data_features.csv")
df_features = pd.read_csv(features_path)
df_features["date"] = pd.to_datetime(df_features["date"])
df_features = df_features.sort_values(["ticker", "date"]).reset_index(drop=True)

print("Shape:", df_features.shape)
print("Tickers:", df_features["ticker"].unique().tolist())
print("Columns:", df_features.columns.tolist())
df_features.head(3)

Shape: (22689, 20)
Tickers: ['AAPL', 'AMZN', 'GOOGL', 'META', 'MSFT', 'MU', 'NFLX', 'NVDA', 'QCOM']
Columns: ['date', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'log_return', 'rsi', 'macd', 'macd_signal', 'macd_hist', 'atr', 'ema_10', 'ema_20', 'ema_50', 'bb_upper', 'bb_middle', 'bb_lower', 'ticker']


,date,open,high,low,close,adj_close,volume,log_return,rsi,macd,macd_signal,macd_hist,atr,ema_10,ema_20,ema_50,bb_upper,bb_middle,bb_lower,ticker
0,2016-03-15,25.990000,26.295000,25.962500,26.145000,23.685328,160270800,0.019894,67.377844,0.384964,0.249910,0.135054,0.516683,25.419239,25.049804,24.913412,26.286548,24.859875,23.433202,AAPL
1,2016-03-16,26.152500,26.577499,26.147499,26.492500,24.000132,153214000,0.013204,70.323953,0.449557,0.289839,0.159717,0.510670,25.614377,25.187203,24.975337,26.541733,24.958000,23.374267,AAPL
2,2016-03-17,26.379999,26.617500,26.240000,26.450001,23.961632,137682800,-0.001605,69.497309,0.491650,0.330202,0.161449,0.501157,25.766309,25.307470,25.033167,26.731654,25.077250,23.422846,AAPL


### 2. Config

In [41]:
LOOKBACK   = 30
HORIZON    = 7
TARGET_COL = "close"
TRAIN_RATIO = 0.7
VAL_RATIO   = 0.15

# feature columns used for scaling and model input
FEATURE_COLS = [c for c in df_features.columns if c not in ["ticker", "date"]]

### 3. Per-Ticker Split → Scale → Sequences

We must process each ticker independently to avoid cross-ticker sequence contamination.

In [ ]:
target_idx = FEATURE_COLS.index(TARGET_COL)

X_train_list, y_train_list = [], []
X_val_list,   y_val_list   = [], []
X_test_list,  y_test_list  = [], []

split_info = []

for ticker, group in df_features.groupby("ticker"):
    g = group.sort_values("date").reset_index(drop=True)

    # 1) chronological split
    train, val, test = train_val_test_split(g, TRAIN_RATIO, VAL_RATIO)

    # 2) scale (fit on train only)
    train_s, val_s, test_s, scaler = scale_splits(train, val, test, FEATURE_COLS)

    # 3) build sequences
    X_tr, y_tr = make_sequences(train_s, LOOKBACK, HORIZON, target_idx)
    X_va, y_va = make_sequences(val_s,   LOOKBACK, HORIZON, target_idx)
    X_te, y_te = make_sequences(test_s,  LOOKBACK, HORIZON, target_idx)

    X_train_list.append(X_tr); y_train_list.append(y_tr)
    X_val_list.append(X_va);   y_val_list.append(y_va)
    X_test_list.append(X_te);  y_test_list.append(y_te)

    split_info.append({
        "ticker": ticker,
        "train_start": train["date"].min(), "train_end": train["date"].max(),
        "val_start":   val["date"].min(),   "val_end":   val["date"].max(),
        "test_start":  test["date"].min(),  "test_end":  test["date"].max(),
        "X_train": X_tr.shape[0], "X_val": X_va.shape[0], "X_test": X_te.shape[0],
    })

# concatenate across tickers
X_train = np.concatenate(X_train_list, axis=0)
y_train = np.concatenate(y_train_list, axis=0)
X_val   = np.concatenate(X_val_list,   axis=0)
y_val   = np.concatenate(y_val_list,   axis=0)
X_test  = np.concatenate(X_test_list,  axis=0)
y_test  = np.concatenate(y_test_list,  axis=0)

print(f"X_train: {X_train.shape}  y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape}    y_val:   {y_val.shape}")
print(f"X_test:  {X_test.shape}   y_test:  {y_test.shape}")

### 4. Validate Splits

In [ ]:
# 4a) Split date ranges per ticker
info_df = pd.DataFrame(split_info)
display(info_df)

In [ ]:
# 4b) No-overlap assertion per ticker
for row in split_info:
    assert row["train_end"] < row["val_start"],  f"{row['ticker']}: train/val overlap!"
    assert row["val_end"]   < row["test_start"], f"{row['ticker']}: val/test overlap!"
print("No overlap confirmed for all tickers")

In [ ]:
# 4c) Visualise split for one ticker
PLOT_TICKER = "NVDA"
g = df_features[df_features["ticker"] == PLOT_TICKER].sort_values("date")
tr, va, te = train_val_test_split(g, TRAIN_RATIO, VAL_RATIO)

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(tr["date"], tr[TARGET_COL], color="blue",   label="Train")
ax.plot(va["date"], va[TARGET_COL], color="orange", label="Val")
ax.plot(te["date"], te[TARGET_COL], color="green",  label="Test")
ax.axvline(tr["date"].max(), color="black", linestyle="--", alpha=0.5)
ax.axvline(va["date"].max(), color="black", linestyle="--", alpha=0.5)
ax.xaxis.set_major_locator(mdates.YearLocator(1))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_title(f"{PLOT_TICKER} — Chronological Train / Val / Test split")
ax.set_ylabel("Close price")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Train: {tr['date'].min().date()} → {tr['date'].max().date()}  ({len(tr)} rows)")
print(f"Val:   {va['date'].min().date()} → {va['date'].max().date()}  ({len(va)} rows)")
print(f"Test:  {te['date'].min().date()} → {te['date'].max().date()}  ({len(te)} rows)")